# Tapping APIs

A site with an API (Application Programming Interface) wants you to have the data it holds.

When we enter a URL in a browser, we typically get back a web page - a formatted document designed for people to read. APIs also use URLs, but rather than delivering visually formatted pages, API URLs deliver structured data that computer programs can easily interpret and process. These specialized API URLs are known as endpoints. Similar to how multiple web pages combine to form a website, multiple endpoints combine to create a complete API. We will learn to construct API calls.

Examples abound:

1. <a href="https://www.census.gov/data/developers/data-sets.html">U.S. Census APIs</a>
2. <a href="https://www.federalregister.gov/developers/documentation/api/v1">Federal Register</a>
3. <a href="https://apps.fas.usda.gov/opendataweb/home">US Agriculture Commodities and Exports</a>
4. <a href="https://www.eia.gov/">U.S. Energy Information Department</a>



Government sites do provide ```CSVs``` for download but their APIs offer more detailed options for data. They are not trying to hide the data.

Private sites might have APIs, but often charge hefty prices for access beyond a basic number of downloads.

The hardest part of tapping APIs is that they ***ALL HAVE DIFFERENT INSTRUCTIONS*** on how to download their data. We'll <a href="https://docs.google.com/presentation/d/158fsOq5FF2qE3dONtwNeePSJyzLRKENlCxRsFPN_ZPg/edit?usp=sharing">learn the best approach to understanding API documentation</a>, including using AI to deconstruct the technical requirements.

In this module, we'll explore different APIs that each build a different skill:

1. Census health data – **building a simple API call.**
2. USDA commodities exports – **using an API key and targeting specific commodities over several years.**
3. Federal Register – **tapping search terms.**
4. Energy Information Administration – **dealing with pagination.**
5. US Spending – **working with rate limits.**

What they all have in common:

1. a base url
2. a query string
3. tied together with query characters ```?``` and ```&```.
4. an API key (<a href="https://docs.google.com/presentation/d/17Cez665ZoDJQaQGdwbY_girCoCgF71Y77aADn88Sv6Q/edit?usp=sharing">different types of authentication</a>)

Combined together these are known as an ```API endpoint```.

You make an ```API call``` (a request) using the ```API endpoint```.




In [18]:
## import libraries
import requests
import pandas as pd
import json




## 1. Census health data – **building a simple API call.**

- <a href="https://www.census.gov/data/developers/data-sets/Health-Insurance-Statistics.html">Census health landing page</a>
- List of <a href="https://api.census.gov/data/timeseries/healthins/sahie/variables.html">possible variables</a>

We want to create a dataframe with the following info for every state in 2021:

1. Total number insured
2. Percent insured
3. Total number uninsured
4. Percent uninsured
5. by Race

## Using API key

<a href="https://api.census.gov/data/key_signup.html">Sign up for an api key</a>

Turns out that the US Census API isn't actually required, but it's best to get one for the following reasons:

1. **Without a key**, your entire IP address is limited to 500 requests per day. 
2. **With a key:** you get 500 requests per day per key (but you can register multiple keys)!

Safeguard your API Key and do not share them. It's good practice to store them in password management in a secure document. I keep all my API keys in a single document.



## Parts to build your API call.

In [23]:
## create a dictionary to know what codes mean
## PERCENT uninsured and insured are by demo groups for selected income range, estimates


target_dict = {
    "NIC_PT": "total_insured",
    "NUI_PT": "total_uninsured",
    "PCTIC_PT": "pct_insured",
    "PCTUI_PT": "pct_uninsured",
    "RACE_DESC": "race_description",
    "RACECAT": "race_categories",
    "STATE": "state",
    "STABREV": "state_name",

}

In [25]:
## we can see just the keys in the dictionary
type(target_dict.keys())
target_dict.keys()

dict_keys(['NIC_PT', 'NUI_PT', 'PCTIC_PT', 'PCTUI_PT', 'RACE_DESC', 'RACECAT', 'STATE', 'STABREV'])

In [27]:
## turn it into a list of keys
list(target_dict.keys())


['NIC_PT',
 'NUI_PT',
 'PCTIC_PT',
 'PCTUI_PT',
 'RACE_DESC',
 'RACECAT',
 'STATE',
 'STABREV']

In [29]:
## pull values into a list
list(target_dict.values())


['total_insured',
 'total_uninsured',
 'pct_insured',
 'pct_uninsured',
 'race_description',
 'race_categories',
 'state',
 'state_name']

In [31]:
## format into api query format
target_vars = ",".join(target_dict.keys())
target_vars

'NIC_PT,NUI_PT,PCTIC_PT,PCTUI_PT,RACE_DESC,RACECAT,STATE,STABREV'

In [33]:
## base url + year
base_url = "https://api.census.gov/data/timeseries/healthins/sahie?get="
target_year = "2021"


In [35]:
## create query string
query_string = f"{target_vars}&time={target_year}"
query_string

'NIC_PT,NUI_PT,PCTIC_PT,PCTUI_PT,RACE_DESC,RACECAT,STATE,STABREV&time=2021'

**Set the key secretly in your notebook:**

We will use `python-dotenv`, a professional way to hide your API key used at Bloomberg and other places.

```python
     from dotenv import load_dotenv
     load_dotenv()  # Reads from .env file 
     import os   ## to handle environment variables
```
You need to ```pip install dotenv``` first.


7. Create a file named ```.env``` using VSCode. **Note:** once you save it and close it, you won't see it anymore.


In [38]:
pip install python-dotenv


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /Users/sandeepjunnarkar/dataProjects/notebook-test-1/.venv/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
import os
from dotenv import load_dotenv


In [42]:
load_dotenv()

True

In [44]:
## pull secret API key into your notebook
census_api_key = os.getenv("CENSUS_API_KEY")

In [46]:
## build api parameter
api_param = f"&key={census_api_key}"
api_param

'&key=e8381112d45fc965ca04930090db2f4b4550c0c6'

In [48]:
## create full API endpoint
endpoint = base_url + query_string + api_param
# endpoint

In [50]:
## get response
response = requests.get(endpoint)
response.status_code

200

In [51]:
## turn response into json
data = response.json()
data 

[['NIC_PT',
  'NUI_PT',
  'PCTIC_PT',
  'PCTUI_PT',
  'RACE_DESC',
  'RACECAT',
  'STATE',
  'STABREV',
  'time'],
 ['241872377',
  '27558913',
  '89.8',
  '10.2',
  'All Races',
  '0',
  '00',
  'US',
  '2021'],
 ['140481313',
  '11001189',
  '92.7',
  '7.3',
  'White alone, not Hispanic or Latino',
  '1',
  '00',
  'US',
  '2021'],
 ['29733477',
  '3647417',
  '89.1',
  '10.9',
  'Black or African American alone, not Hispanic or Latino',
  '2',
  '00',
  'US',
  '2021'],
 ['3548525', '469887', '88.3', '11.7', 'All Races', '0', '01', 'AL', '2021'],
 ['530325', '80538', '86.8', '13.2', 'All Races', '0', '02', 'AK', '2021'],
 ['5016330', '758331', '86.9', '13.1', 'All Races', '0', '04', 'AZ', '2021'],
 ['2158741', '268334', '88.9', '11.1', 'All Races', '0', '05', 'AR', '2021'],
 ['30088120', '2640001', '91.9', '8.1', 'All Races', '0', '06', 'CA', '2021'],
 ['4406138', '452402', '90.7', '9.3', 'All Races', '0', '08', 'CO', '2021'],
 ['2702690', '172986', '94.0', '6.0', 'All Races', '0', 

In [53]:
## create dataframe
df = pd.DataFrame(data[1:], columns = data[0])
df

,NIC_PT,NUI_PT,PCTIC_PT,PCTUI_PT,RACE_DESC,RACECAT,STATE,STABREV,time
0,241872377,27558913,89.8,10.2,All Races,0,00,US,2021
1,140481313,11001189,92.7,7.3,"White alone, not Hispanic or Latino",1,00,US,2021
2,29733477,3647417,89.1,10.9,"Black or African American alone, not Hispanic ...",2,00,US,2021
3,3548525,469887,88.3,11.7,All Races,0,01,AL,2021
4,530325,80538,86.8,13.2,All Races,0,02,AK,2021
...,...,...,...,...,...,...,...,...,...
3554,5604,1209,82.3,17.7,All Races,0,56,WY,2021
3555,29682,5465,84.5,15.5,All Races,0,56,WY,2021
3556,16469,3204,83.7,16.3,All Races,0,56,WY,2021
3557,14758,2489,85.6,14.4,All Races,0,56,WY,2021


In [56]:
df.query("STATE == '01'")

,NIC_PT,NUI_PT,PCTIC_PT,PCTUI_PT,RACE_DESC,RACECAT,STATE,STABREV,time
3,3548525,469887,88.3,11.7,All Races,0,01,AL,2021
86,2293718,254385,90.0,10.0,"White alone, not Hispanic or Latino",1,01,AL,2021
87,921007,135939,87.1,12.9,"Black or African American alone, not Hispanic ...",2,01,AL,2021
180,163769,59615,73.3,26.7,Hispanic or Latino (any race),3,01,AL,2021
181,13274,2587,83.7,16.3,"American Indian and Alaska Native alone, not H...",4,01,AL,2021
...,...,...,...,...,...,...,...,...,...
478,27074,4080,86.9,13.1,All Races,0,01,AL,2021
479,163945,18030,90.1,9.9,All Races,0,01,AL,2021
481,10345,1385,88.2,11.8,All Races,0,01,AL,2021
482,7012,931,88.3,11.7,All Races,0,01,AL,2021


In [58]:
target_values = list(target_dict.values())
target_values

['total_insured',
 'total_uninsured',
 'pct_insured',
 'pct_uninsured',
 'race_description',
 'race_categories',
 'state',
 'state_name']

In [60]:
## rename column headers with more meaning full headers
df = df.rename(columns = target_dict)
df

,total_insured,total_uninsured,pct_insured,pct_uninsured,race_description,race_categories,state,state_name,time
0,241872377,27558913,89.8,10.2,All Races,0,00,US,2021
1,140481313,11001189,92.7,7.3,"White alone, not Hispanic or Latino",1,00,US,2021
2,29733477,3647417,89.1,10.9,"Black or African American alone, not Hispanic ...",2,00,US,2021
3,3548525,469887,88.3,11.7,All Races,0,01,AL,2021
4,530325,80538,86.8,13.2,All Races,0,02,AK,2021
...,...,...,...,...,...,...,...,...,...
3554,5604,1209,82.3,17.7,All Races,0,56,WY,2021
3555,29682,5465,84.5,15.5,All Races,0,56,WY,2021
3556,16469,3204,83.7,16.3,All Races,0,56,WY,2021
3557,14758,2489,85.6,14.4,All Races,0,56,WY,2021


In [64]:
## iterate throught several years
df_list = []
for target_year in range(2020,2024):
    print(f"Tapping {target_year}")
    query_string = f"{target_vars}&time={target_year}"
    endpoint = base_url + query_string + api_param
    response = requests.get(endpoint)
    data = response.json()
    df = pd.DataFrame(data[1:], columns = data[0])
    df_list.append(df)

print("done with all target years")
    
    
    
    

Tapping 2020
Tapping 2021
Tapping 2022
Tapping 2023
done with all target years


In [68]:
## length
len(df_list)

4

In [72]:
## place in df
df = pd.concat(df_list)
df

,NIC_PT,NUI_PT,PCTIC_PT,PCTUI_PT,RACE_DESC,RACECAT,STATE,STABREV,time
0,239173287,27895218,89.6,10.4,All Races,0,00,US,2020
1,139345483,11539058,92.4,7.6,"White alone, not Hispanic or Latino",1,00,US,2020
2,29079112,3900919,88.2,11.8,"Black or African American alone, not Hispanic ...",2,00,US,2020
3,44950517,10180150,81.5,18.5,Hispanic or Latino (any race),3,00,US,2020
4,3437796,460936,88.2,11.8,All Races,0,01,AL,2020
...,...,...,...,...,...,...,...,...,...
3555,48126,5610,89.6,10.4,All Races,0,49,UT,2023
3556,10829,903,92.3,7.7,All Races,0,51,VA,2023
3557,54518,2853,95.0,5.0,All Races,0,51,VA,2023
3558,9548,791,92.3,7.7,All Races,0,53,WA,2023


### 2. USDA commodities exports – **using an API key and targeting specific commodities over several years.**

- <a href="https://apps.fas.usda.gov/opendataweb/home">USDA APIs endpoints</a>
- Get an <a href="https://apps.fas.usda.gov/opendataweb/home">API key</a>

We want to create a dataframe with exports between 2020-2022 to all countries for the following commodities:

1. All Wheat
2. Oats
3. Cuts of Beef
4. Cuts of Pork

In [31]:
## load api key secretly


In [32]:
## find the parts to build your API call

headers = {
    'accept': 'application/json',
    'X-Api-Key': ADD HERE
}


SyntaxError: invalid syntax (3922311065.py, line 5)

In [ ]:
## TARGET COMMODITIES

In [ ]:
#BUILD base and end url


In [ ]:
## iterate through multi year


In [ ]:
##SEE ALL DATA - DO NOT RUN THIS CELL
all_data

In [ ]:
pd.DataFrame(all_data)

In [ ]:
response.json()

In [ ]:
headers = {
    'accept': 'application/json',
    'X-Api-Key': 'lx8NMObcTL9PSZSCZzJyykInlqLEx5WZZKtbGITw',
}

In [ ]:
url = "https://api.fas.usda.gov/api/esr/exports/commodityCode/601/allCountries/marketYear/2023"

In [ ]:


response = requests.get(url = url, headers=headers)

In [ ]:
response.json()

In [ ]:
response.json()

In [ ]:
countries_response = requests.get("https://api.fas.usda.gov/api/esr/countries", headers=headers)
countries_response.json()

In [ ]:
## now let's put into get requests
## we check the response status code
response = requests.get(url = com_url, headers = headers)
response.json()

### Get endpoint and test out on a single commodity


In [ ]:
## your end point here


In [ ]:
## now let's put into get requests
## we check the response status code


In [ ]:
## let's store our response into an object called data


In [ ]:
## convert that list of dicts into a dataframe called df


In [ ]:
## Now iterate through all our target items



In [ ]:
## endpoint templates



In [ ]:
## iterate to get all the data


In [ ]:
## call list


In [ ]:
## concat into single df


In [ ]:
## call df


In [ ]:
## confirm we have all our target commodities


### 3. <a id="federal"></a>Federal Register – **tapping search terms.**

We have decades of <a href="https://docs.google.com/spreadsheets/d/130WeumbMZjcoRP4D-1uJ7bM0aKBZzt4N/edit?usp=sharing&ouid=112307892189798608417&rtpof=true&sd=true">SBA Excel files</a> that detail loans given to small businesses to recover after climate disasters. The only information we have about the type of disasters are codes in one of the columns that look like:

- CA-00279
- IL-00051
- NC-00099
- CA-00288
- LA-00079

The <a href="https://www.federalregister.gov/">Federal Register</a> allows us to search for what these codes stand for. But we can't search for nearly a thousand such disaster codes. When we try to scrape the site, it warns us to use the API instead.

Federal Register <a href="https://www.federalregister.gov/developers/documentation/api/v1#/Federal%20Register%20Documents/get_documents__format_">API documentation</a>

## find the end point

https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=LA-00079

https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=PA-00115

#### Test on single endpoint after figuring out how to build API call

In [50]:
## endpoint
endpoint = "https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=HI-00067"

In [52]:
## get data
response = requests.get(endpoint)
data = response.json()
data

{'description': 'Documents filed under agency docket HI-00067',
 'count': 1,
 'total_pages': 1,
 'results': [{'title': 'Administrative Declaration of a Disaster for the State of Hawaii',
   'type': 'Notice',
   'abstract': 'This is a notice of an Administrative declaration of a disaster for the State of HAWAII dated 01/28/2022. Incident: Severe Storms, Flooding and Landslides. Incident Period: 12/05/2021 through 12/10/2021.',
   'document_number': '2022-02191',
   'html_url': 'https://www.federalregister.gov/documents/2022/02/03/2022-02191/administrative-declaration-of-a-disaster-for-the-state-of-hawaii',
   'pdf_url': 'https://www.govinfo.gov/content/pkg/FR-2022-02-03/pdf/2022-02191.pdf',
   'public_inspection_pdf_url': 'https://public-inspection.federalregister.gov/2022-02191.pdf?1643809525',
   'publication_date': '2022-02-03',
   'agencies': [{'raw_name': 'SMALL BUSINESS ADMINISTRATION',
     'name': 'Small Business Administration',
     'id': 468,
     'url': 'https://www.federalr

In [ ]:
## length


In [ ]:
## what type is it?


In [ ]:
## GET COUNT


In [94]:
## GET RESULTS
incident_all = data["results"][0]["abstract"]
incident_all

'This is a notice of an Administrative declaration of a disaster for the State of Utah dated 10/01/2021. Incident: Severe Storms and Flooding. Incident Period: 08/01/2021.'

In [102]:
incident_all.split("Incident")[1].replace(": ", "").replace(". ", "")

'Severe Storms and Flooding'

In [ ]:
## type


In [ ]:
## LEN OF LIST


In [ ]:
## NARROW OUR FIELD


In [ ]:
## targeting incidents


In [64]:
pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]━━ 1/2 [openpyxl]

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /Users/sandeepjunnarkar/dataProjects/notebook-test-1/.venv/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [72]:
## 

dfraw = pd.read_excel("sba22.xlsx", sheet_name = "FY22 Home", skiprows=4 )
dfraw

,SBA Physical Declaration Number,SBA EIDL Declaration Number,FEMA Disaster Number,SBA Disaster Number,Damaged Property City Name,Damaged Property Zip Code,Damaged Property County/Parish Name,Damaged Property State Code,Total Verified Loss,Verified Loss Real Estate,Verified Loss Content,Total Approved Loan Amount,Approved Amount Real Estate,Approved Amount Content
0,17206,17207,NaN,UT-00087,CEDAR CITY,84720,IRON,UT,184281.40,122359.40,61922.0,56800.0,36900.0,19900.0
1,17206,17207,NaN,UT-00087,CEDAR CITY,84721,IRON,UT,119187.00,119187.00,0.0,119200.0,119200.0,0.0
2,17206,17207,NaN,UT-00087,ENOCH,84721,IRON,UT,183515.00,128139.00,55376.0,183700.0,128300.0,55400.0
3,17215,17216,NaN,PA-00115,EPHRATA,17522,LANCASTER,PA,49917.00,0.00,49917.0,NaN,0.0,0.0
4,17215,17216,NaN,PA-00115,MAYTOWN,17550,LANCASTER,PA,NaN,NaN,NaN,NaN,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066,17655,17656,4675.0,FL-00179,PORT ST LUCIE,34984,SAINT LUCIE,FL,69566.38,58571.38,10995.0,NaN,0.0,0.0
3067,17655,17656,4675.0,FL-00179,PORT SAINT LUCIE,34984,SAINT LUCIE,FL,NaN,NaN,NaN,NaN,0.0,0.0
3068,17655,17656,4675.0,FL-00179,PORT ST LUCIE,34986,SAINT LUCIE,FL,71026.95,65676.95,5350.0,NaN,0.0,0.0
3069,17655,17656,4675.0,FL-00179,FORT PIERCE,34987,SAINT LUCIE,FL,NaN,NaN,NaN,NaN,0.0,0.0


In [84]:
disaster_codes = list(dfraw["SBA Disaster Number"].unique())
print(disaster_codes)

['UT-00087', 'PA-00115', 'PA-00116', 'AZ-00076', 'GA-00131', 'MD-00043', 'MS-00136', 'CA-00348', 'OR-00126', 'CA-00349', 'CT-00054', 'KY-00086', 'GA-00132', 'CA-00351', 'FL-00170', 'CA-00344', 'CA-00352', 'KY-00087', 'AL-00125', 'AR-00120', 'CO-00136', 'WA-00100', 'TN-00132', 'NY-00207', 'DE-00028', 'HI-00067', 'VA-00096', 'WA-00103', 'PA-00119', 'AL-00126', 'FL-00173', 'PR-00040', 'OK-00155', 'SC-00078', 'AR-00121', 'GA-00137', 'KS-00150', 'TX-00628', 'FL-00171', 'TX-00627', 'NM-00080', 'TN-00136', 'KY-00091', 'MT-00159', 'OK-00157', 'IL-00069', 'MS-00145', 'IN-00077', 'MI-00108', 'TN-00137', 'MN-00095', 'MN-00096', 'MS-00144', 'KY-00093', 'MO-00113', 'CA-00361', 'PA-00120', 'CA-00362', 'IN-00078', 'MN-00099', 'AZ-00083', 'TX-00644', 'WV-00057', 'AZ-00084', 'AZ-00085', 'PR-00042', 'AK-00055', 'FL-00178', 'IL-00073', 'FL-00179']


### Iterate through entire list of codes

In [86]:
## Normally will take from df as a list
## build disaster code list
print(disaster_codes)

['UT-00087', 'PA-00115', 'PA-00116', 'AZ-00076', 'GA-00131', 'MD-00043', 'MS-00136', 'CA-00348', 'OR-00126', 'CA-00349', 'CT-00054', 'KY-00086', 'GA-00132', 'CA-00351', 'FL-00170', 'CA-00344', 'CA-00352', 'KY-00087', 'AL-00125', 'AR-00120', 'CO-00136', 'WA-00100', 'TN-00132', 'NY-00207', 'DE-00028', 'HI-00067', 'VA-00096', 'WA-00103', 'PA-00119', 'AL-00126', 'FL-00173', 'PR-00040', 'OK-00155', 'SC-00078', 'AR-00121', 'GA-00137', 'KS-00150', 'TX-00628', 'FL-00171', 'TX-00627', 'NM-00080', 'TN-00136', 'KY-00091', 'MT-00159', 'OK-00157', 'IL-00069', 'MS-00145', 'IN-00077', 'MI-00108', 'TN-00137', 'MN-00095', 'MN-00096', 'MS-00144', 'KY-00093', 'MO-00113', 'CA-00361', 'PA-00120', 'CA-00362', 'IN-00078', 'MN-00099', 'AZ-00083', 'TX-00644', 'WV-00057', 'AZ-00084', 'AZ-00085', 'PR-00042', 'AK-00055', 'FL-00178', 'IL-00073', 'FL-00179']


In [88]:
## provide base url
base_url = "https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]="

In [106]:
## ITERATE THROUGH ALL
broken_endpoints =[]
incidents_list = []

## iterate through all endpoints

for disaster_code in disaster_codes:
    endpoint = base_url + disaster_code
    print(endpoint)
    try:
        response = requests.get(endpoint)
        data = response.json()
        incident_text = data["results"][0]["abstract"]
        incident_description = incident_text.split("Incident")[1].replace(": ", "").replace(". ", "")
        incidents_list.append({"SBA Disaster Number": disaster_code,
                             "disaster_description": incident_description})
    except:
        print(f"The {endpoint} endpoint was broken")
        broken_endpoints.append(endpoint)
        

https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=UT-00087
https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=PA-00115
https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=PA-00116
https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=AZ-00076
https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=GA-00131
https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=MD-00043
https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=MS-00136
https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=CA-00348
https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=OR-00126
The https://www.federalregister.gov/api/v1/documents.json?per_page=20&conditions[docket_id]=OR-00126 endpoint was broken
https:

In [108]:
## call list
incidents_list

[{'SBA Disaster Number': 'UT-00087',
  'disaster_description': 'Severe Storms and Flooding'},
 {'SBA Disaster Number': 'PA-00115',
  'disaster_description': 'Remnants of Tropical Depression Ida'},
 {'SBA Disaster Number': 'PA-00116',
  'disaster_description': 'Remnants of Hurricane Ida'},
 {'SBA Disaster Number': 'AZ-00076',
  'disaster_description': 'Severe Storms, Flooding and Flash Flooding'},
 {'SBA Disaster Number': 'GA-00131',
  'disaster_description': 'Severe Flood Damage from Hurricane Ida'},
 {'SBA Disaster Number': 'MD-00043',
  'disaster_description': 'Remnants of Tropical Storm Ida'},
 {'SBA Disaster Number': 'MS-00136', 'disaster_description': 'Hurricane Ida'},
 {'SBA Disaster Number': 'CA-00348', 'disaster_description': 'Hopkins Fire'},
 {'SBA Disaster Number': 'CA-00349',
  'disaster_description': 'Brannan Island Fire'},
 {'SBA Disaster Number': 'CT-00054',
  'disaster_description': 'Remnants of Hurricane Ida'},
 {'SBA Disaster Number': 'KY-00086',
  'disaster_descriptio

In [110]:
## DATA FRAME IT
df_incidents = pd.DataFrame(incidents_list)
df_incidents

,SBA Disaster Number,disaster_description
0,UT-00087,Severe Storms and Flooding
1,PA-00115,Remnants of Tropical Depression Ida
2,PA-00116,Remnants of Hurricane Ida
3,AZ-00076,"Severe Storms, Flooding and Flash Flooding"
4,GA-00131,Severe Flood Damage from Hurricane Ida
...,...,...
56,PR-00042,Hurricane Fiona
57,AK-00055,"Severe Storm, Flooding, and Landslides"
58,FL-00178,Hurricane Ian
59,IL-00073,Apartment Building Explosion


In [112]:
df = pd.merge(df_incidents, dfraw)
df

,SBA Disaster Number,disaster_description,SBA Physical Declaration Number,SBA EIDL Declaration Number,FEMA Disaster Number,Damaged Property City Name,Damaged Property Zip Code,Damaged Property County/Parish Name,Damaged Property State Code,Total Verified Loss,Verified Loss Real Estate,Verified Loss Content,Total Approved Loan Amount,Approved Amount Real Estate,Approved Amount Content
0,UT-00087,Severe Storms and Flooding,17206,17207,NaN,CEDAR CITY,84720,IRON,UT,184281.40,122359.40,61922.0,56800.0,36900.0,19900.0
1,UT-00087,Severe Storms and Flooding,17206,17207,NaN,CEDAR CITY,84721,IRON,UT,119187.00,119187.00,0.0,119200.0,119200.0,0.0
2,UT-00087,Severe Storms and Flooding,17206,17207,NaN,ENOCH,84721,IRON,UT,183515.00,128139.00,55376.0,183700.0,128300.0,55400.0
3,PA-00115,Remnants of Tropical Depression Ida,17215,17216,NaN,EPHRATA,17522,LANCASTER,PA,49917.00,0.00,49917.0,NaN,0.0,0.0
4,PA-00115,Remnants of Tropical Depression Ida,17215,17216,NaN,MAYTOWN,17550,LANCASTER,PA,NaN,NaN,NaN,NaN,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2812,FL-00179,Hurricane Ian,17655,17656,4675.0,PORT ST LUCIE,34984,SAINT LUCIE,FL,69566.38,58571.38,10995.0,NaN,0.0,0.0
2813,FL-00179,Hurricane Ian,17655,17656,4675.0,PORT SAINT LUCIE,34984,SAINT LUCIE,FL,NaN,NaN,NaN,NaN,0.0,0.0
2814,FL-00179,Hurricane Ian,17655,17656,4675.0,PORT ST LUCIE,34986,SAINT LUCIE,FL,71026.95,65676.95,5350.0,NaN,0.0,0.0
2815,FL-00179,Hurricane Ian,17655,17656,4675.0,FORT PIERCE,34987,SAINT LUCIE,FL,NaN,NaN,NaN,NaN,0.0,0.0
